# 1.1 - Math Foundations: The Basics

**Module 1 - Math Prerequisites** | Netsetos GenAI Engineering

This notebook builds the four numerical primitives that every LLM runs on, in bare NumPy: representing words as vectors, ranking by cosine similarity, turning scores into probabilities with softmax, and steering that distribution with temperature. It closes with a 30-line cell that chains all four into one retrieve-then-generate pass.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - Install NumPy

Everything in this notebook is pure NumPy - no deep-learning framework, no API calls. This first cell just makes sure the one dependency is present.

In [ ]:
!pip install -q numpy

**Explanation**

A one-line environment-prep cell. `-q` keeps pip quiet so the output stays clean; if NumPy is already installed (it usually is in Colab) this is a fast no-op.

**How the code works - step by step**
- **`!pip install -q numpy`** - shell-out (the `!`) that installs NumPy quietly.

**In one line:** guarantee NumPy exists before we import it.

**What you'll see:** (no output - environment prep)

## Setup - Import NumPy

One import powers the entire notebook. NumPy gives us arrays, dot products, and vector norms - the only machinery the four skills need.

In [ ]:
import numpy as np

**Explanation**

The standard import that binds NumPy to the conventional `np` alias used in every cell below.

**How the code works - step by step**
- **`import numpy as np`** - the array library; every vector, `np.dot`, `np.exp`, and `np.linalg.norm` call later depends on it.

**In one line:** load NumPy as `np`.

**What you'll see:** (no output - environment prep)

## 1 - Words as vectors, and the king - man + woman analogy

A word embedding is just a list of numbers, and meaning lives in the geometry between those lists. This cell hand-builds 5 tiny 4-dimensional vectors, then shows the famous analogy trick: doing arithmetic on the vectors moves you to a new point in meaning-space.

In [ ]:
words = {
    'king':   np.array([0.9, 0.1, 0.8, 0.0]),
    'queen':  np.array([0.9, 0.9, 0.8, 0.0]),
    'man':    np.array([0.1, 0.1, 0.5, 0.0]),
    'woman':  np.array([0.1, 0.9, 0.5, 0.0]),
    'pizza':  np.array([0.0, 0.0, 0.0, 0.9]),
}

result = words['king'] - words['man'] + words['woman']
print(f'king - man + woman = {result}')
print(f"queen              = {words['queen']}")

**Explanation**

This cell is a hand-crafted demo, not a trained model - the numbers are chosen so the geometry works out cleanly. The four dimensions loosely stand for concepts like royalty, gender, and food; subtracting `man` and adding `woman` cancels the 'male' component and injects the 'female' one, landing almost exactly on `queen`.

**How the code works - step by step**
- **`words = {...}`** - a dictionary mapping five words to 4-D NumPy arrays; each array is a toy embedding.
- **`words['king'] - words['man'] + words['woman']`** - element-wise vector arithmetic: strip the 'man' direction, add the 'woman' direction.
- **the two `print`s** - show the computed result next to the real `queen` vector so you can eyeball the match.

**In one line:** meaning is a vector, so analogies become addition and subtraction.

**What you'll see:** Two printed rows. The computed `king - man + woman` comes out to `[0.9 0.9 0.8 0. ]`, which matches the `queen` row exactly - the analogy lands on the right word.

## 2 - Cosine similarity: ranking by direction, not size

To find which document best answers a query, you compare the angle between their vectors, not the distance. Cosine similarity does exactly that, and this cell uses it to rank five documents against one search query.

In [ ]:
def cosine_similarity(a, b, eps=1e-6):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps)

documents = {
    'Python tutorial':  np.array([0.9, 0.8, 0.1, 0.0]),
    'JavaScript guide': np.array([0.8, 0.7, 0.2, 0.0]),
    'Pasta recipe':     np.array([0.0, 0.1, 0.9, 0.8]),
    'Pizza recipe':     np.array([0.0, 0.1, 0.8, 0.9]),
    'Machine learning': np.array([0.7, 0.9, 0.3, 0.0]),
}
query = np.array([0.85, 0.75, 0.15, 0.0])

ranked = sorted(documents.items(), key=lambda kv: cosine_similarity(query, kv[1]), reverse=True)
for doc, vec in ranked:
    sim = cosine_similarity(query, vec)
    print(f'  {doc:18s} cos={sim:.3f}')

**Explanation**

`cosine_similarity` is the workhorse of every retrieval system in the course - it measures how aligned two vectors point, ignoring their lengths. The `eps` term is a small guard so a zero-length vector never divides by zero. The rest of the cell is a miniature search engine: score every document against the query, then sort high-to-low.

**How the code works - step by step**
- **`cosine_similarity(a, b, eps=1e-6)`** - returns `dot(a,b)` divided by the product of the two norms (plus `eps`); result runs from -1 to 1, where 1 is identical direction.
- **`documents = {...}` and `query`** - five 4-D doc vectors and one query vector, all hand-set so coding docs cluster together and food docs cluster elsewhere.
- **`sorted(..., key=lambda kv: cosine_similarity(query, kv[1]), reverse=True)`** - scores each doc against the query and orders them best-first.
- **the `for` loop** - prints each document with its cosine score, formatted to 3 decimals.

**In one line:** score by angle, then sort - that is retrieval.

**What you'll see:** Five rows printed best-match-first. The three coding-related docs (Python tutorial, JavaScript guide, Machine learning) score highest (cos near 1.0), while the two food recipes score near 0 - the query about programming correctly ranks the code docs on top.

## 3 - Softmax: turning raw scores into a probability distribution

A model outputs raw scores (logits), not probabilities. Softmax converts any list of real numbers into positive values that sum to 1, so they can be read as 'chance of each token'. This cell implements the numerically stable version and runs it on three candidate next-tokens.

In [ ]:
def softmax(logits):
    shifted = logits - np.max(logits)
    e = np.exp(shifted)
    return e / e.sum()

tokens = ['cat', 'dog', 'fish']
logits = np.array([2.0, 1.0, 0.1])
probs = softmax(logits)
for tok, p in zip(tokens, probs):
    print(f'  {tok:5s} {p:.3f}')
print(f'  sum = {probs.sum():.6f}')

**Explanation**

This is the exact softmax used inside every LLM's output layer. The one subtlety is the `logits - np.max(logits)` shift: it changes nothing mathematically but keeps `np.exp` from overflowing on large logits - the reason this is called the *stable* softmax. Bigger logits get exponentially more probability mass.

**How the code works - step by step**
- **`shifted = logits - np.max(logits)`** - subtract the max so the largest exponent is `exp(0)=1`; prevents overflow.
- **`e = np.exp(shifted)`** - exponentiate, making everything positive and amplifying gaps between scores.
- **`return e / e.sum()`** - normalize so the outputs add to exactly 1.
- **the `for` loop + final `print`** - map each token to its probability and confirm the distribution sums to 1.

**In one line:** exponentiate the scores, then normalize - that is a probability distribution.

**What you'll see:** Three token probabilities - roughly `cat 0.659`, `dog 0.242`, `fish 0.099` - and a final line showing `sum = 1.000000`. The highest logit (cat) claims the most mass.

## 4 - Temperature: sharpening or flattening the distribution

Same logits, same softmax - but dividing the logits by a temperature `T` before softmax controls how confident the output is. Low `T` makes the model decisive; high `T` makes it adventurous. This cell sweeps four temperatures and draws a bar so you can *see* the distribution flatten.

In [ ]:
def softmax_with_temp(logits, T=1.0):
    scaled = logits / T
    shifted = scaled - np.max(scaled)
    e = np.exp(shifted)
    return e / e.sum()

tokens = ['cat', 'dog', 'fish']
logits = np.array([2.0, 1.0, 0.1])
for T in [0.1, 0.5, 1.0, 2.0]:
    p = softmax_with_temp(logits, T=T)
    bar_cat = '#' * int(p[0] * 40)
    print(f'  T={T:3.1f}  cat={p[0]:.3f}  dog={p[1]:.3f}  fish={p[2]:.3f}  {bar_cat}')

**Explanation**

This is the `temperature` knob you set on every generation API, implemented from scratch. Dividing logits by a small `T` blows up the gaps between them (near-greedy, almost all mass on the top token); dividing by a large `T` shrinks the gaps (closer to uniform, more random). It is the single line `logits / T` layered on top of the stable softmax from the previous cell.

**How the code works - step by step**
- **`softmax_with_temp(logits, T=1.0)`** - identical to the earlier softmax except it first computes `scaled = logits / T`.
- **`scaled - np.max(scaled)`** - the same overflow-safe shift, applied to the temperature-scaled logits.
- **the `for T in [0.1, 0.5, 1.0, 2.0]` loop** - recomputes probabilities at each temperature.
- **`bar_cat = '#' * int(p[0] * 40)`** - a text bar whose length tracks the 'cat' probability, so you watch it shrink as `T` rises.

**In one line:** divide logits by T before softmax - small T = sharp, large T = flat.

**What you'll see:** Four rows, one per temperature. At `T=0.1` cat is near-certain (a long `#` bar, probability close to 1); as `T` climbs to 2.0 the three probabilities converge and cat's bar shrinks - the distribution visibly flattens.

## 5 - Production pipeline: all four primitives in one pass

Retrieval and generation are the same two operations you just learned, chained. This final cell picks the best document by cosine, then samples a next-token from a temperature-scaled softmax - a retrieve-then-generate loop in about 30 lines.

In [ ]:
query_vec = np.array([0.85, 0.75, 0.15, 0.0])

docs = {'py_tut': np.array([0.9,0.8,0.1,0.0]),
        'pasta':  np.array([0.0,0.1,0.9,0.8])}
top = max(docs, key=lambda k: cosine_similarity(query_vec, docs[k]))

logits = np.array([2.0, 1.0, 0.1])

def sample(logits, T=0.7):
    p = np.exp((logits - logits.max()) / T)
    p = p / p.sum()
    return np.random.choice(len(p), p=p)

np.random.seed(42)
token_idx = sample(logits, T=0.7)
print(f'Retrieved: {top}, generated token idx: {token_idx}')

**Explanation**

A capstone cell that reuses `cosine_similarity` from Skill 2 and folds Skills 3 and 4 into a `sample` helper. Unlike the earlier cells that printed the full distribution, this one actually *draws* a token from it with `np.random.choice`, weighted by the softmax probabilities - which is what real generation does. The fixed seed makes that random draw reproducible.

**How the code works - step by step**
- **`top = max(docs, key=lambda k: cosine_similarity(query_vec, docs[k]))`** - the retrieval step: pick the single best-matching document by cosine.
- **`sample(logits, T=0.7)`** - computes a temperature-0.7 softmax, then returns a token index drawn from that distribution via `np.random.choice`.
- **`np.random.seed(42)`** - pins the RNG so the sampled token is identical on every run.
- **`token_idx = sample(logits, T=0.7)`** and the `print` - run one generation step and report both the retrieved doc and the sampled token index.

**In one line:** cosine to retrieve, softmax-sample to generate - the whole RAG loop in miniature.

**What you'll see:** A single line: `Retrieved: py_tut, generated token idx: <n>` - the coding query correctly retrieves the Python doc, and a next-token index is sampled from the temperature-0.7 distribution (fixed by the seed).

You now have the four primitives that recur through the whole course: vectors carry meaning, cosine ranks it, softmax turns raw scores into a probability distribution, and temperature reshapes that distribution before you sample from it. The final cell shows they are not separate tricks - retrieval and generation are the same two operations (cosine, then softmax-sample) stitched together. Every embedding, RAG, and sampling lesson later in the course is a scaled-up version of what you just ran in bare NumPy.